# Prepare eligible adjusted-price histories

Read `Data/eligible/data_mapped.csv`. Join nonoverlapping ISINs using known continuity links or stable `CompanyId` and `SecurityId` pairs. Also join successive ISINs with a shared `Internal code` when they meet on adjacent trading sessions and agree on the mapped ICB labels.

In [1]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd

data_dir = Path("Data") if Path("Data").is_dir() else Path("../Data")
source = data_dir / "eligible" / "data_mapped.csv"
output = source.with_name("price_histories.csv")
icb_columns = [f"icb_{level}_name" for level in ("industry", "supersector", "sector", "subsector")]
columns = ["Date", "ISIN", "Name", "Sector", "CompanyId", "SecurityId", "Internal code",
           "AdjustedPrice", "Turnover", "Currency", *icb_columns]
data = pd.read_csv(source, sep=";", decimal=",", usecols=columns, parse_dates=["Date"])
data = data.sort_values(["Date", "ISIN"], kind="stable")
assert data[["Date", "ISIN"]].notna().all().all()
assert data["Currency"].dropna().eq("NOK").all()
sessions = pd.DatetimeIndex(data["Date"].unique(), name="Date")
session_number = pd.Series(np.arange(len(sessions)), index=sessions)


def identifiers(values: pd.Series) -> tuple:
    return tuple(sorted(values.dropna().unique()))


profiles = data.groupby("ISIN").agg(
    first=("Date", "min"), last=("Date", "max"), company=("CompanyId", identifiers),
    security=("SecurityId", identifiers), internal=("Internal code", identifiers),
)
labels = data.groupby("ISIN")[icb_columns].last()
histories = {isin: {isin} for isin in profiles.index}


def join_histories(isins: list[str], *, adjacent: bool = False) -> bool:
    """Join compatible ISIN spans while keeping simultaneous classes separate."""
    members = set().union(*(histories[isin] for isin in isins))
    spans = profiles.loc[sorted(members)].sort_values("first")
    starts = spans["first"].iloc[1:].map(session_number).to_numpy()
    ends = spans["last"].iloc[:-1].map(session_number).to_numpy()
    if not (starts > ends).all():
        return False
    if adjacent and (not (starts == ends + 1).all() or len(labels.loc[spans.index].drop_duplicates()) != 1):
        return False
    for isin in members:
        histories[isin] = members
    return True


known_links = [
    ["NO0003043309", "NO0013536151"],  # Kongsberg
    ["BMG368191034", "BMG3682E1012", "BMG3682E1921", "CY0200352116"],  # Frontline
    ["NO0005668905", "NO0012470089"],  # Tomra
]
for aliases in known_links:
    found = [isin for isin in aliases if isin in profiles.index]
    if len(found) > 1:
        assert join_histories(found), found

for keys, adjacent in [(["company", "security"], False), (["internal"], True)]:
    stable = profiles.loc[profiles[keys].map(len).eq(1).all(axis=1)]
    for _, group in stable.groupby(keys):
        if len(group) > 1:
            join_histories(group.index.tolist(), adjacent=adjacent)

stock_by_isin = {}
for isin, members in histories.items():
    spans = profiles.loc[sorted(members)].sort_values("last")
    assert spans["first"].iloc[1:].gt(spans["last"].shift().iloc[1:]).all(), isin
    stock_by_isin[isin] = spans.index[-1]
data["stock_id"] = data["ISIN"].map(stock_by_isin)
assert data.groupby(["stock_id", "Date"])["ISIN"].nunique().eq(1).all()
print(f"Linked {len(stock_by_isin)} ISINs into {data['stock_id'].nunique()} price histories.")

Linked 140 ISINs into 119 price histories.


## Keep one share class per company

Group histories transitively by shared nonmissing CompanyId or Internal code. Keep the class with the most usable adjusted-price dates, then the highest summed NOK turnover, then the lowest stock ID. Export one row per retained history and trading session with gaps left blank.

In [2]:
keys = ["stock_id", "Date"]
grouped = data.groupby(keys)
records = data.drop_duplicates(keys).set_index(keys).sort_index()
records["duplicate_rows"] = grouped.size() - 1
records["price_conflict"] = grouped["AdjustedPrice"].nunique(dropna=False).gt(1)
price = records["AdjustedPrice"]
records["invalid_price"] = ~(np.isfinite(price) & price.gt(0))
records["AdjustedPrice"] = price.where(~(records["price_conflict"] | records["invalid_price"]))
turnover = records["Turnover"]
records["Turnover"] = turnover.where(
    grouped["Turnover"].nunique(dropna=False).eq(1) & np.isfinite(turnover) & turnover.ge(0)
)

# Use issuer identifiers for class selection without combining class prices.
issuer_members = {stock_id: {stock_id} for stock_id in records.index.unique("stock_id")}
for identifier in ("CompanyId", "Internal code"):
    links = data[["stock_id", identifier]].dropna().drop_duplicates()
    for _, stock_group in links.groupby(identifier)["stock_id"]:
        members = set().union(*(issuer_members[stock_id] for stock_id in stock_group))
        for stock_id in members:
            issuer_members[stock_id] = members

class_ranking = records.groupby(level="stock_id").agg(
    usable_dates=("AdjustedPrice", "count"), total_turnover=("Turnover", "sum")
).reset_index()
class_ranking["issuer"] = class_ranking["stock_id"].map(
    {stock_id: min(members) for stock_id, members in issuer_members.items()}
)
class_ranking = class_ranking.sort_values(
    ["usable_dates", "total_turnover", "stock_id"], ascending=[False, False, True]
)
retained = class_ranking.drop_duplicates("issuer")
keep = retained["stock_id"]
contested = class_ranking.loc[class_ranking["issuer"].duplicated(keep=False)].copy()
contested["retained"] = contested["stock_id"].isin(keep)
display(contested)

records = records.loc[records.index.get_level_values("stock_id").isin(keep)]
by_stock = data.loc[data["stock_id"].isin(keep)].groupby("stock_id")
metadata = by_stock[["Name", "CompanyId", "Sector", *icb_columns]].last()
metadata["sector_changed"] = by_stock["Sector"].nunique().gt(1)
grid = pd.MultiIndex.from_product([metadata.index, sessions], names=keys)
fields = ["ISIN", "AdjustedPrice", "Turnover", "duplicate_rows", "price_conflict", "invalid_price"]
panel = records[fields].reindex(grid).join(metadata).reset_index()
panel["duplicate_rows"] = panel["duplicate_rows"].fillna(0).astype(int)
for field in ("price_conflict", "invalid_price"):
    panel[field] = panel[field].eq(True)

previous = records.groupby(level="stock_id")[["ISIN", "AdjustedPrice"]].shift()
changed = previous["ISIN"].notna() & previous["ISIN"].ne(records["ISIN"])
transitions = records.loc[changed, ["ISIN", "AdjustedPrice"]].join(previous.add_prefix("previous_"))
transitions["price_change"] = transitions["AdjustedPrice"] / transitions["previous_AdjustedPrice"] - 1
print(f"{len(transitions)} ISIN transitions; {transitions['price_change'].abs().ge(.20).sum()} changes of 20% or more.")
display(transitions)

assert not panel.duplicated(keys).any()
assert panel["ISIN"].notna().sum() == len(records)
with NamedTemporaryFile(dir=output.parent, suffix=".tmp", delete=False) as dst:
    temporary = Path(dst.name)
try:
    panel.to_csv(temporary, index=False)
    temporary.replace(output)
finally:
    temporary.unlink(missing_ok=True)
print(f"Wrote {len(panel):,} rows for {len(metadata)} histories across {len(sessions):,} sessions to {output}.")

21 ISIN transitions; 0 changes of 20% or more.
Wrote 288,650 rows for 115 histories across 2,510 sessions to Data/eligible/price_histories.csv.


,stock_id,usable_dates,total_turnover,issuer,retained
26,NO0003080608,2510,2.618373e+10,NO0003080608,True
78,NO0010571698,2510,8.859349e+09,NO0010571698,True
98,NO0010736879,2509,9.835731e+10,NO0003028904,True
79,NO0010576010,2493,3.264604e+09,NO0010571698,False
31,NO0003399909,2481,5.443283e+09,NO0003399909,True
32,NO0003399917,2353,1.938428e+09,NO0003399909,False
15,NO0003028904,2294,1.291514e+11,NO0003028904,False
100,NO0010771215,123,8.558964e+07,NO0003080608,False


,,ISIN,AdjustedPrice,previous_ISIN,previous_AdjustedPrice,price_change
stock_id,Date,,,,,
BMG0451H2087,2024-05-07,BMG0451H2087,23.393883,BMG0451H1170,24.982515,-0.063590
CY0200352116,2023-01-13,CY0200352116,94.242034,BMG3682E1921,94.950621,-0.007463
NO0010161896,2021-07-02,NO0010161896,130.091252,NO0010031479,129.812759,0.002145
NO0012470089,2022-05-27,NO0012470089,177.226119,NO0005668905,164.171307,0.079519
NO0012483207,2022-04-01,NO0012483207,68.427359,NO0006390004,68.087693,0.004989
NO0012555459,2022-06-20,NO0012555459,24.294515,NO0010379779,24.841017,-0.022000
NO0012953720,2023-07-03,NO0012953720,1.042184,NO0010283211,1.154203,-0.097053
NO0013033795,2023-10-04,NO0013033795,10.227998,NO0010689326,10.042034,0.018519
NO0013119255,2024-01-23,NO0013119255,4.490000,NO0010284318,4.500000,-0.002222
